In [2]:
import sys
import os
import pandas as pd

# Add the parent directory of 'app' to the Python path
sys.path.append(os.path.abspath('D:/GitHub/personal_finances_analysis'))

from src.ofx_dataprep import OfxDataPrep
from src.llm_finance import LLMFinance

# In root path we have the folders input, output and processed
root_path = "D:\GitHub\_DADOS\personal_finances"
input_path = os.path.join(root_path, "input")

Pre-processing and Saving Data
- bank account 
- credit card transactions

In [3]:
dataprep = OfxDataPrep()
transactions = dataprep.read_and_prep_data(data_path=input_path)

transactions.to_csv(os.path.join(root_path, "output", "transactions_prep.csv"), index=False, encoding='utf-8')

Read processed data

In [4]:
transactions = pd.read_csv(os.path.join(root_path, "output", "transactions_prep.csv"), encoding='utf-8')
transactions.shape

(639, 12)

Bank Account Transactions
- is_installment = False = There isn't transactions with installment = True
- Is not good idea control how much you received trought bank "in" transactions because you schedulled a transaction of R$4k from Santander to Nubank, and everytime the value is lower or higher the scheduled value. So in this way you need to check your SANTANDER account, every month, to collect this information and fill the spreadsheet
- In transactions doesn't have valuable information for the analysis, for now I will concentrade in the rest of transactions

In [16]:
df_bank = transactions.query("type == 'Bank'").copy()
df_bank.shape

(160, 12)

In [19]:
df_bank_in = df_bank.query("in_out == 'in'").copy()
df_bank_out = df_bank.query("in_out == 'out'").copy()

df_bank_in.shape, df_bank_out.shape

((52, 12), (108, 12))

In [23]:
# BANK IN
df_bank_in.groupby('memo')['amount'].sum().sort_values(ascending=False)

memo
Transferência recebida pelo Pix - MATHEUS GONZALEZ EUGENIO - •••.210.918-•• - BCO SANTANDER (BRASIL) S.A. (0033) Agência: 218 Conta: 1015097-6         47863.33
Resgate RDB                                                                                                                                            18062.57
Transferência recebida pelo Pix - ADILSON AMARO DA SILVA - •••.770.118-•• - BCO SANTANDER (BRASIL) S.A. (0033) Agência: 2973 Conta: 1044078-9           1068.00
Transferência recebida pelo Pix - ELISANGELA CRISTINA TOTTORO - •••.325.258-•• - BCO SANTANDER (BRASIL) S.A. (0033) Agência: 197 Conta: 1090366-8        291.00
Transferência Recebida - Scarlett Tottoro Santos - •••.102.508-•• - NU PAGAMENTOS - IP (0260) Agência: 1 Conta: 23041583-5                               260.00
Crédito em conta                                                                                                                                         172.47
Transferência de saldo NuInvest    

In [53]:
display(transactions.shape)

display(transactions['institution'].value_counts())

display(transactions['type'].value_counts(normalize=True))

display(transactions['in_out'].value_counts(normalize=True))

display(transactions.groupby(['institution', 'type', 'account_id', 'is_installment', 'in_out']).size().unstack().fillna(0).astype(int))

important_columns = ['year_month', 'amount', 'memo', 'type', 'in_out', 'is_installment', 'institution']

(639, 12)

institution
NU PAGAMENTOS S.A.    639
Name: count, dtype: int64

type
Credit Card    0.749609
Bank           0.250391
Name: proportion, dtype: float64

in_out
out    0.890454
in     0.109546
Name: proportion, dtype: float64

in_out                                                                              in  \
institution        type        account_id                           is_installment       
NU PAGAMENTOS S.A. Bank        2209842-6                            False           52   
                   Credit Card 5a43cccd-bdaf-4211-a5c5-bc45326df942 False           18   
                                                                    True             0   

in_out                                                                              out  
institution        type        account_id                           is_installment       
NU PAGAMENTOS S.A. Bank        2209842-6                            False           108  
                   Credit Card 5a43cccd-bdaf-4211-a5c5-bc45326df942 False           353  
                                                                    True            108

In [57]:
bank_transactions = transactions.query("type == 'Bank'")[important_columns].copy()
cc_transactions = transactions.query("type == 'Credit Card'")[important_columns].copy()

bank_transactions.shape, cc_transactions.shape

((160, 7), (479, 7))

In [4]:
# Group messages to get unique messages
trans_grouped = transactions.groupby('memo')['id'].apply(list).reset_index(name='id_list')
trans_grouped.shape

(331, 2)

In [ ]:
# Models
# "llama3-8b-8192"
# "gemma2-9b-it"
# "deepseek-r1-distill-llama-70b"

# Classifying transactions
models = ["llama3-8b-8192", "gemma2-9b-it", "deepseek-r1-distill-llama-70b"]

for model_name in models:
    llm_finance = LLMFinance(model_name=model_name)
    transactions_classified = llm_finance.classify_batch(trans_grouped['memo'].head())
    print(model_name)
    display(transactions_classified)

llama3-8b-8192


,memo,category
0,1a99,Compras
1,1a99 - Parcela 1/3,Compras Parceladas
2,1a99 - Parcela 2/3,Compras Parceladas
3,1a99 - Parcela 3/3,Compras Parceladas
4,1password,Assinaturas e Serviços


gemma2-9b-it


,memo,category
0,1a99,Compras Parceladas
1,1a99 - Parcela 1/3,Compras Parceladas
2,1a99 - Parcela 2/3,Compras Parceladas
3,1a99 - Parcela 3/3,Compras Parceladas
4,1password,Assinaturas e Serviços


deepseek-r1-distill-llama-70b


,memo,category
0,1a99,"<think>\nOkay, so I need to figure out the cor..."
1,1a99 - Parcela 1/3,"<think>\nOkay, I'm looking at the transaction ..."
2,1a99 - Parcela 2/3,"<think>\nOk, eu sou um iniciante na análise de..."
3,1a99 - Parcela 3/3,"<think>\nOkay, so I have this transaction to c..."
4,1password,"Okay, so I'm trying to figure out the right ca..."


In [6]:
llm_finance = LLMFinance(model_name="gemma2-9b-it")
transactions_classified = llm_finance.classify_batch(trans_grouped['memo'])
display(transactions_classified)

[Throttle] Waiting for 51.67 seconds to respect RPM limit...
[Throttle] Waiting for 43.26 seconds to respect RPM limit...
[Throttle] Waiting for 33.07 seconds to respect RPM limit...
[Throttle] Waiting for 22.82 seconds to respect RPM limit...
[Throttle] Waiting for 12.42 seconds to respect RPM limit...
[Throttle] Waiting for 2.21 seconds to respect RPM limit...


,memo,category
0,1a99,Compras
1,1a99 - Parcela 1/3,Compras Parceladas
2,1a99 - Parcela 2/3,Compras Parceladas
3,1a99 - Parcela 3/3,Compras Parceladas
4,1password,Assinaturas e Serviços
...,...,...
326,Vindi *Kinvo - Parcela 11/12,Compras Parceladas
327,Vindi *Kinvo - Parcela 12/12,Compras Parceladas
328,Wellhub Gympass Br Gym,Assinaturas e Serviços
329,Zig*Madalena Cervejari,Restaurantes


In [9]:
#transactions.shape
transactions_classified['category'].value_counts()

category
Compras Parceladas               99
Compras                          73
Assinaturas e Serviços           34
Transferências para terceiros    32
Restaurantes                     32
Mercado                          13
Transporte                       11
Fatura                           10
Receitas                          7
Moradia                           7
Saúde                             5
Investimento                      3
Telefone                          3
Horti Fruti                       2
Name: count, dtype: int64

In [13]:
transactions_classified.query("category == 'Transferências para terceiros'").values

array([['29002030rodrigo', 'Transferências para terceiros'],
       ['Alexandre da Silva Cam', 'Transferências para terceiros'],
       ['Claudiasantosda', 'Transferências para terceiros'],
       ['Ifd*52.730.088 Rita Re', 'Transferências para terceiros'],
       ['Ifd*Giovanna Dias Schi', 'Transferências para terceiros'],
       ['Maria Neide Alves dos', 'Transferências para terceiros'],
       ['Mauriciofeliciano', 'Transferências para terceiros'],
       ['Transferência enviada pelo Pix - Aparecido Donizeti Eugênio - •••.543.058-•• - NU PAGAMENTOS - IP (0260) Agência: 1 Conta: 8920037-5',
        'Transferências para terceiros'],
       ['Transferência enviada pelo Pix - Aparecido Donizeti Eugênio - •••.543.058-•• - Nubank (0260) Agência: 1 Conta: 8920037-5',
        'Transferências para terceiros'],
       ['Transferência enviada pelo Pix - Brenno do Nascimento Eugenio - •••.648.108-•• - BCO SOFISA S.A. (0637) Agência: 1 Conta: 651271-4',
        'Transferências para terceiros'],


- Alterar fluxo e separar transações de cartão de crédito das de CC, se necessário, criar dois prompts para realizar a classificação.
- Como calibrar a temperatura do modelo? Quais outros parâmetros posso utilizar para isso?
- Como avaliar o melhor modelo para a classificação? De olho não dá
- Traduzir o prompt para inglês mantendo apenas as categorias em português, o resultado melhora?
- Quais outras estratégias de prompts posso utilizar? few-shot? quais outras?
- Criar dashboard no streamlit: tabela que permita atualizar e salvar as categorias das transações, total por categoria e lista de transações por categoria.